# NB24 — A2 random graph for P3 x spatial with the spatial graph shuffled (CPU; rebuilds the cache)

Downloads ds004504 (version pinned), runs the fixed preamble and the feature cache for all 88 subjects, then on the selection and confirmation folds trains P3 x spatial (reference) and P3 with a degree-preserving shuffle of the spatial graph (A2), on identical splits and seeds.

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
SHA = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
sh(f'pip install -q -e "{SRC}[preprocess,dev,stats]"', log=f'{WORK}/pip_install.log')
PY = f'cd {SRC} && {sys.executable} -m eegrep'
print('git sha', SHA)

In [ ]:
sh(f'cd {SRC} && {sys.executable} -m pytest -q -p no:cacheprovider tests/test_audit.py tests/test_graphs.py tests/test_ablations.py', log=f'{WORK}/pytest_nb24.txt')

In [ ]:
sh(f'{PY} download --dest /tmp/ds004504', log=f'{WORK}/download.log')
shutil.copy('/tmp/ds004504/participants.tsv', f'{WORK}/participants.tsv')
shutil.copy('/tmp/ds004504/download_manifest.json', f'{WORK}/download_manifest.json')

In [ ]:
sh(f'{PY} preprocess --bids /tmp/ds004504 --out /tmp/preproc --n-jobs 4', log=f'{WORK}/preprocess.log')
shutil.copy('/tmp/preproc/preprocess_log.jsonl', f'{WORK}/preprocess_log.jsonl')

In [ ]:
sh(f'{PY} cache --preproc /tmp/preproc --participants {WORK}/participants.tsv --out {WORK}/cache', log=f'{WORK}/cache.log')
sh(f'{PY} qc --cache {WORK}/cache --logs {WORK}/preprocess_log.jsonl --out {WORK}/qc_report.json', log=f'{WORK}/qc.log')

In [ ]:
sh(f'{PY} grid --cache {WORK}/cache --folds {SRC}/splits/folds_ds004504.json --pipelines P3 --edges spatial --tag main --out {WORK}/ref_selection.jsonl', log=f'{WORK}/ref_selection.log')
sh(f'{PY} ablate --cache {WORK}/cache --folds {SRC}/splits/folds_ds004504.json --participants {WORK}/participants.tsv --cells P3xspatial --which A2_random_graph --out {WORK}/a2_selection.jsonl', log=f'{WORK}/a2_selection.log')

In [ ]:
sh(f'{PY} grid --cache {WORK}/cache --folds {SRC}/splits/folds_ds004504_confirm.json --pipelines P3 --edges spatial --tag confirm --out {WORK}/ref_confirmation.jsonl', log=f'{WORK}/ref_confirmation.log')
sh(f'{PY} ablate --cache {WORK}/cache --folds {SRC}/splits/folds_ds004504_confirm.json --participants {WORK}/participants.tsv --cells P3xspatial --which A2_random_graph --out {WORK}/a2_confirmation.jsonl', log=f'{WORK}/a2_confirmation.log')

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': SHA, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)